In [16]:
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopandas.tools import geocode

df = pd.read_csv('/Users/aayushkumbhare/Desktop/cafo-contamination/cleaned_2022.csv')
df = df.reset_index()
print(df)
        
         


     index  Unnamed: 0                operator_name        physical_street  \
0        0           0           John Machado Dairy  22495 W China Camp RD   
1        1           1                          NaN                    NaN   
2        2           2             O'Banion Ranches     15775 S Indiana RD   
3        3           3                 Borges Farms      5148 W Grayson RD   
4        4           4            Sierra Rose Dairy  9022 Aguas Frias Road   
..     ...         ...                          ...                    ...   
428    428         428        Dairy Central Heifers      10119 W August RD   
429    429         429              Van Steyn Dairy       11823 Carroll RD   
430    430         430          Vogt Holstein Dairy   7831 South Capay Ave   
431    431         431  Lourenco/Wickstrom Dairy #4       16126 Sunset AVE   
432    432         432     Tony Oliveira, Jr. Dairy    27506 W Gun Club RD   

    physical_city physical_county physical_zip mailing_street m

In [18]:
#cleaning dataframe (removing NAN's) 
required = ['physical_street', 'physical_city', 'physical_zip', 'milk_cows_avg_number', 'nitrogen_from_manure_lbs', 'nitrogen_after_ammonia_losses_lbs']
to_drop = ['mailing_street', 'mailing_city', 'mailing_state', 'mailing_zip', 'permit_number', 'reporting_period', 'groundwater_monitoring']
df.dropna(subset=required, inplace=True)
df = df.dropna(axis=1, how='all')
#df = df.drop(columns=to_drop, inplace=True)

print(df.head)
print(len(df.columns))

<bound method NDFrame.head of      index  Unnamed: 0                operator_name        physical_street  \
2        2           2             O'Banion Ranches     15775 S Indiana RD   
3        3           3                 Borges Farms      5148 W Grayson RD   
4        4           4            Sierra Rose Dairy  9022 Aguas Frias Road   
6        6           6          Tony Lopes Dairy LP      27500 W Bunker RD   
7        7           7   Alden Peterson & Sons Inc.     1431 N Central AVE   
..     ...         ...                          ...                    ...   
426    426         426               Goedhart Dairy    7084 County Road 31   
428    428         428        Dairy Central Heifers      10119 W August RD   
429    429         429              Van Steyn Dairy       11823 Carroll RD   
430    430         430          Vogt Holstein Dairy   7831 South Capay Ave   
431    431         431  Lourenco/Wickstrom Dairy #4       16126 Sunset AVE   

    physical_city physical_county

In [20]:

def get_address(df):
    for idx, row in df.iterrows():
        street = row['physical_street']
        city = row['physical_city']
        zipcode = row['physical_zip']
        state = 'CA'

        address = f"{street}, {city}, {state} {zipcode}"
        df.loc[idx, 'address'] = address
        #print(f"Added address for row {idx}")

    return df

df = get_address(df)
print(df.columns)    
    

Index(['index', 'Unnamed: 0', 'operator_name', 'physical_street',
       'physical_city', 'physical_county', 'physical_zip', 'mailing_street',
       'mailing_city', 'mailing_state', 'mailing_zip', 'operator_phone',
       'date_placed_in_operation', 'basin_plan_designation',
       'assessor_parcels', 'milk_cows_open_confinement',
       'milk_cows_under_roof', 'milk_cows_max_number', 'milk_cows_avg_number',
       'dry_cows_open_confinement', 'dry_cows_under_roof',
       'dry_cows_max_number', 'dry_cows_avg_number',
       'bred_heifers_open_confinement', 'bred_heifers_under_roof',
       'bred_heifers_max_number', 'bred_heifers_avg_number',
       'heifers_7_14_mo_open_confinement', 'heifers_7_14_mo_under_roof',
       'heifers_7_14_mo_max_number', 'heifers_7_14_mo_avg_number',
       'calves_4_6_mo_open_confinement', 'calves_4_6_mo_under_roof',
       'calves_4_6_mo_max_number', 'calves_4_6_mo_avg_number',
       'calves_0_3_mo_open_confinement', 'calves_0_3_mo_under_roof',
      

In [22]:
#geocoding lat-long
import os
from dotenv import load_dotenv
import json
import boto3

load_dotenv()


geocode_client = boto3.client('geo-places', region_name='us-west-2')

def get_coordinates(df):
    if 'latitude' not in df.columns:
        df['latitude'] = None

    if 'longitude' not in df.columns:
        df['longitude'] = None
    for idx, row in df.iterrows():
        address = row['address']

        try:
            response = geocode_client.geocode(
                QueryText=address,
                MaxResults=1
            )
        except Exception as e:
            print(f"Error adding coordinate, error: {e}")
        if response['ResultItems']:
            place = response['ResultItems'][0]
            coordinates = place['Position']
            print(f"Latitude: {coordinates[1]}; Longitude: {coordinates[0]}; address: {address}")


            df.at[idx, 'longitude'] = coordinates[0]
            df.at[idx, 'latitude'] = coordinates[1] 
        
            print(f"Added coordinates for address {address}")
        else:
            print(f"Coordinates for {address} not added")
    
    return df

df = get_coordinates(df)
df.to_csv('2022_final.csv')



Latitude: 37.05626; Longitude: -120.59865; address: 15775 S Indiana RD, Dos Palos, CA 93620
Added coordinates for address 15775 S Indiana RD, Dos Palos, CA 93620
Latitude: 37.5658; Longitude: -121.08767; address: 5148 W Grayson RD, Modesto, CA 95358
Added coordinates for address 5148 W Grayson RD, Modesto, CA 95358
Latitude: 39.63197; Longitude: -121.85899; address: 9022 Aguas Frias Road, Chico, CA 95928
Added coordinates for address 9022 Aguas Frias Road, Chico, CA 95928
Latitude: 37.17334; Longitude: -120.98684; address: 27500 W Bunker RD, Gustine, CA 95322
Added coordinates for address 27500 W Bunker RD, Gustine, CA 95322
Latitude: 37.50553; Longitude: -120.96076; address: 1431 N Central AVE, Turlock, CA 95380
Added coordinates for address 1431 N Central AVE, Turlock, CA 95380
Latitude: 37.44016; Longitude: -120.97421; address: 5608 S Morgan RD, Turlock, CA 95380
Added coordinates for address 5608 S Morgan RD, Turlock, CA 95380
Latitude: 37.36427; Longitude: -120.86506; address: 207